## 1 Definition of pseudo-concepts

Pseudo-concepts are computationally derived spatial regions inside lesion masks designed to approximate structural aspects of dermatological analysis.

Unlike true clinical concepts, pseudo-concepts:
- do not require expert annotation
- can be generated automatically
- provide interpretable spatial partitions of lesions

Let:
- I  = dermoscopic image
- M  = lesion segmentation mask
- S  = saliency map produced by an XAI method

Pseudo-concepts define a set of regions: $ P = {p_1, p_2, ..., p_n}$
such as $ p_i ⊂ M $
⋃ p_i = M $

Each region renables measurement of a saliency concentration inside region $p_i$.




## 2 Proposed pseudo-concept regions

## 2.1 Border and core regions (B for Border)
Lesion borders play an important role in dermoscopy because irregular or poorly defined borders are associated with malignant lesions.

Border = outer band
Core = Interior
Enable to distinguish between explanations around lesion boundaries and central lesion structures.

## 2.2 Spatial sectors (A for Asymetric)
Asymmetry as a dermatological concept relates to lesion symetrtic dysimilarity along two perpendicular axis. This can be easily computed into a pseudo-concept of lesion halves or quadrant.

This allows detection of cases where explanations focus on a single region of the lesion rather than being distributed.

## 2.3 Lesion size (D for Diameter)
Although images are not uniformely scalable, using pixels, an equivalent diameter can be defined, and therefore a lesion area and a major axis lenght.
These lesion geometric descriptors allow  analysis of how lesion size influences explanation behavior.


## 3. Pseudo-concept saliency metrics
Given a saliency map S a mask M and a pseudo-concept region $p_i$

### 3.1 Regional saliency ratio: This measures how much saliency lies in region $p_i$.

 $SR_i = \frac{\sum(S \subset p_i)}{\sum(S \subset M)}$

### 3.2 Saliency Entropy
This is an indice of explainations' distribution vs. concentration.
$ H = - \sum SR_i log(SR_i) $

### 3.3 Border emphasis index
This metric measures whether explanations emphasise lesion boundaries.


### Implementation - to be tested

In [ ]:
# load mask

mask = cv2.imread(mask_path, 0)
mask = (mask > 0).astype(np.uint8)

# border / core computation
from scipy.ndimage import distance_transform_edt

dist = distance_transform_edt(mask)

max_dist = dist.max()
normalized = dist / max_dist

border = (normalized <= 0.25) & (mask == 1)
core = (normalized > 0.25) & (mask == 1)

# lesion centroid computation
import numpy as np
ys, xs = np.where(mask == 1)

cy = ys.mean()
cx = xs.mean()
print(f"Lesion centroid: ({cx:.2f}, {cy:.2f})")

# generate quadrants region
h, w = mask.shape
Y, X = np.indices((h, w))

q1 = (mask == 1) & (Y < cy) & (X < cx)
q2 = (mask == 1) & (Y < cy) & (X >= cx)
q3 = (mask == 1) & (Y >= cy) & (X < cx)
q4 = (mask == 1) & (Y >= cy) & (X >= cx)

## Saliency map per region
def saliency_ratio(saliency, region):

    total = saliency[mask == 1].sum()
    region_sal = saliency[region].sum()

    return region_sal / total

sr_border = saliency_ratio(S, border)
sr_q1 = saliency_ratio(S, q1)